Load the data, import libraries, show the sample

In [1]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
players = pd.read_csv(url)
players.sample(10)

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
123,Beginner,False,e74c60a92c0100e7240be56d66969db85856152b048c63...,7.1,Arash,Male,17,NaN,NaN
67,Amateur,True,18936844e06b6c7871dce06384e2d142dd86756941641e...,17.2,Kyrie,Male,14,NaN,NaN
110,Amateur,True,827f9a82b47cdf21098bc7bb1d4241550dc6146ba9ae8b...,0.0,Sean,Male,22,NaN,NaN
48,Veteran,True,b3510c708bd50bf9f75e6e02bb6fe14edb705e0ea671ee...,12.5,Isidore,Agender,27,NaN,NaN
40,Regular,True,0d4d71be33e2bc7266ee4983002bd930f69d304288a866...,5.6,Winslow,Male,17,NaN,NaN
99,Pro,True,8eae13e0f5823d9c48fe7b89695ea231f96f644f664bcc...,0.5,Gray,Male,12,NaN,NaN
133,Beginner,True,7b5c323f9a9e21b6bcc2a8c431e96f9f2155c7b4318a68...,0.0,Xenos,Two-Spirited,17,NaN,NaN
14,Veteran,True,6f9acf8ea9956fe817895c78d10e1e25c11aba335a451e...,0.0,Niamh,Non-binary,17,NaN,NaN
83,Beginner,False,4535690bb1e71073b59d049bd8055089c25573510f13aa...,0.0,Akira,Male,28,NaN,NaN
109,Amateur,True,fe218a05c6c3fc6326f4f151e8cb75a2a9fa29e22b110d...,0.1,Fatima,Male,17,NaN,NaN


Extract the data to 3 colunms

In [9]:
player_selected = players[["subscribe","played_hours","age"]]
player_selected

,subscribe,played_hours,age
0,True,30.3,9
1,True,3.8,17
2,False,0.0,17
3,True,0.7,21
4,True,0.1,21
...,...,...,...
191,True,0.0,17
192,False,0.3,22
193,False,0.0,17
194,False,2.3,17


Set the random seed

In [10]:
np.random.seed(1)

Split the train set and test set

In [11]:
player_train, player_test = train_test_split(
    player_selected, train_size=0.75, stratify=player_selected["subscribe"]
)
player_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 147 entries, 20 to 127
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   subscribe     147 non-null    bool   
 1   played_hours  147 non-null    float64
 2   age           147 non-null    int64  
dtypes: bool(1), float64(1), int64(1)
memory usage: 3.6 KB


standarize the data

In [12]:
player_preprocessor = make_column_transformer(
    (StandardScaler(), ["played_hours", "age"]),
)

train the classifier with the k = 3

In [13]:
X = player_train[["played_hours", "age"]]
y = player_train["subscribe"]
knn = KNeighborsClassifier(n_neighbors=3)
knn_pipeline = make_pipeline(player_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['played_hours', 'age'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=3))])

predict the test set with the model and test the score

In [14]:
player_test["predicted"] = knn_pipeline.predict(player_test[["played_hours", "age"]])
player_test[["subscribe", "predicted"]]
knn_pipeline.score(
    player_test[["played_hours", "age"]],
    player_test["subscribe"]
)

0.5714285714285714

Cross validation

In [15]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()
player_tune_pipe = make_pipeline(player_preprocessor, knn)

parameter_grid = {
    "kneighborsclassifier__n_neighbors": range(2, 20, 1),
}

player_tune_grid = GridSearchCV(
    estimator=player_tune_pipe,
    param_grid=parameter_grid,
    cv=10
)

player_tune_grid.fit(
    player_train[["played_hours", "age"]],
    player_train["subscribe"]
)

accuracies_grid = pd.DataFrame(player_tune_grid.cv_results_)
accuracies_grid.info()

accuracies_grid["sem_test_score"] = accuracies_grid["std_test_score"] / 10**(1/2)
accuracies_grid = (
    accuracies_grid[[
        "param_kneighborsclassifier__n_neighbors",
        "mean_test_score",
        "sem_test_score"
    ]]
    .rename(columns={"param_kneighborsclassifier__n_neighbors": "n_neighbors"})
)
accuracies_grid

accuracy_vs_k = alt.Chart(accuracies_grid).mark_line(point=True).encode(
    x=alt.X("n_neighbors").title("Neighbors"),
    y=alt.Y("mean_test_score")
        .scale(zero=False)
        .title("Accuracy estimate")
)

accuracy_vs_k

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   mean_fit_time                            18 non-null     float64
 1   std_fit_time                             18 non-null     float64
 2   mean_score_time                          18 non-null     float64
 3   std_score_time                           18 non-null     float64
 4   param_kneighborsclassifier__n_neighbors  18 non-null     int64  
 5   params                                   18 non-null     object 
 6   split0_test_score                        18 non-null     float64
 7   split1_test_score                        18 non-null     float64
 8   split2_test_score                        18 non-null     float64
 9   split3_test_score                        18 non-null     float64
 10  split4_test_score                        18 non-null

alt.Chart(...)

From the plot we can see k = 8 might be the best classifier.

Train the model with k = 8 

In [26]:
X = players[["age", "played_hours"]]      
y = players["subscribe"]                  
exp = players["experience"]               

# Train/test split
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exp,
    test_size=0.25,
    stratify=y,          
    random_state=2025,
)

knn_pipe = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=8)
)

knn_pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=8))])

see the test data

In [31]:
y_pred = knn_pipe.predict(X_test)

results = pd.DataFrame({
    "experience": exp_test.values,
    "y_true": y_test.values,
    "y_pred": y_pred,
})

results["correct"] = results["y_true"] == results["y_pred"]
acc_by_exp = (
    results
    .groupby("experience")["correct"]
    .mean()
    .reset_index()
    .rename(columns={"correct": "accuracy"})
)

acc_by_exp

,experience,accuracy
0,Amateur,0.642857
1,Beginner,0.555556
2,Pro,1.000000
3,Regular,0.888889
4,Veteran,0.714286


Visualization of the accuracy toward different kinds of players

In [32]:
acc_plot = alt.Chart(acc_by_exp).mark_bar().encode(
    x=alt.X("experience:N", title="Experience level"),
    y=alt.Y("accuracy:Q", title="Prediction accuracy"),
    tooltip=[
        "experience",
        alt.Tooltip("accuracy:Q", title="Accuracy", format=".2f")
    ]
).properties(
    title="KNN (k = 8) accuracy by experience level"
)
acc_plot

alt.Chart(...)